In [ ]:
# !pip install lightgbm

In [1]:
from IPython.core.interactiveshell import InteractiveShell
InteractiveShell.ast_node_interactivity = "all"
from sklearn.metrics import classification_report
from sklearn.metrics import roc_auc_score
from sklearn.preprocessing import OrdinalEncoder
from sklearn.model_selection  import train_test_split
import numpy as np
import pandas as pd
# import warnings
# warnings.filterwarnings("ignore")
# 读入数据
raw_data= pd.read_csv('保险数据_第一期.csv')
# 缺失值(缺失值占比很少，直接删除)
df= raw_data.dropna().reset_index(drop=True)
# 分割标签
labels= df.pop("resp_flag")
# 数据类型拆分
cat_cols= df.select_dtypes(include= ["object"])   # 分类型
num_cols= df.select_dtypes(exclude= ["object"])   # 数值型
# 数据编码
encoder= OrdinalEncoder()
cat_trans= encoder.fit_transform(cat_cols)
df_cat= pd.DataFrame(cat_trans, columns= cat_cols.columns)
# 数据合并
data= pd.concat([df_cat, num_cols], axis=1)
# 分割测试集
Xtrain, Xtest, Ytrain, Ytest= train_test_split(data, labels, test_size= 0.3, stratify= labels, random_state= 42)

# Sklearn部分实现LightGBM

In [2]:
from sklearn.ensemble import HistGradientBoostingClassifier

HGBC= HistGradientBoostingClassifier()
HGBC.fit(Xtrain, Ytrain)

print("AUC: ", roc_auc_score(Ytest, HGBC.predict_proba(Xtest)[:, 1]))

HistGradientBoostingClassifier()

AUC:  0.712222968129737


# LightGBM库

## 基础使用

In [8]:
# sklearn接口（不推荐）
from lightgbm import LGBMClassifier
lgb= LGBMClassifier()
lgb.fit(Xtrain, Ytrain)

print("AUC: ", roc_auc_score(Ytest, lgb.predict_proba(Xtest)[:, 1]))

[LightGBM] [Info] Number of positive: 10963, number of negative: 16379
[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.002853 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 2062
[LightGBM] [Info] Number of data points in the train set: 27342, number of used features: 49
[LightGBM] [Info] [binary:BoostFromScore]: pavg=0.400958 -> initscore=-0.401474
[LightGBM] [Info] Start training from score -0.401474


LGBMClassifier()

AUC:  0.7120481346854272


In [2]:
# lgb接口
import lightgbm

train= lightgbm.Dataset(Xtrain, label= Ytrain)

lgb = lightgbm.train(params={}, train_set= train)

print("AUC: ", roc_auc_score(Ytest, lgb.predict(Xtest)))

[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.003568 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 2062
[LightGBM] [Info] Number of data points in the train set: 27342, number of used features: 49
[LightGBM] [Info] Start training from score 0.400958
AUC:  0.7088783916050837


In [10]:
# 数据集转换为二进制保存，加快加载速度
train.save_binary('train.bin')

[LightGBM] [Info] Saving data to binary file train.bin


In [11]:
train1= lightgbm.Dataset('train.bin')

lgb1 = lightgbm.train(params={}, train_set= train1)

[LightGBM] [Info] Load from binary file train.bin
[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.002640 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 2062
[LightGBM] [Info] Number of data points in the train set: 27342, number of used features: 49
[LightGBM] [Info] Start training from score 0.400958


In [ ]:
# # 附加参数
# metric= ""   评估指标{rmse, quantile, poisson, auc, binary_loglogg, multi_logloss}
# device_type= 'cpu'   {'cpu', 'gpu', 'cuda'}
# max_cat_to_onehot= 4, 自动哑变量转换的类别数
# use_quantized_grad= False,  训练时是否使用梯度量化, 只支持cpu,cuda

## 进阶使用

### 数据集

In [3]:
# 数据集
Xtrain1, Xtest1, Ytrain1, Ytest1= train_test_split(Xtrain, Ytrain, test_size= 0.3, stratify= Ytrain, random_state= 42)

train1= lightgbm.Dataset(data= Xtrain1, label= Ytrain1,
                         categorical_feature= ['GEND', 'AART'],
                         params= {'max_bin': 255},   # 附加参数
                         free_raw_data=False)
test1= lightgbm.Dataset(data= Xtest1, label= Ytest1, reference=train1)  # reference参数只能传入训练集

In [ ]:
# # 附加参数（数据集）
# max_bin= 255   最大分箱数
# max_bin_by_feature= None   每个特征的最大箱数
# min_data_in_bin= 3   每个箱子内的最小样本数
# bin_construct_sample_cnt= 200000   构造箱子的最大样本数
# two_round= False   数据集两轮加载，只支持cpu,cuda

### 训练

In [5]:
# 训练
params= {'num_leaves': 31, 'objective': 'binary'}
params['metric'] = 'auc'
params['verbosity'] = 3

lgb = lightgbm.train(params=params, train_set= train1, valid_sets= test1,
                     num_boost_round= 200,
                     callbacks=[lightgbm.early_stopping(stopping_rounds=5, min_delta=0)],   # min_delta至少提升多少数值
                     )

print("AUC: ", roc_auc_score(Ytest, lgb.predict(Xtest)))

[LightGBM] [Info] Number of positive: 7674, number of negative: 11465
[LightGBM] [Debug] Dataset::GetMultiBinFromSparseFeatures: sparse rate 0.854411
[LightGBM] [Debug] Dataset::GetMultiBinFromAllFeatures: sparse rate 0.425948
[LightGBM] [Debug] init for col-wise cost 0.001504 seconds, init for row-wise cost 0.002886 seconds
[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.001990 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Debug] Using Sparse Multi-Val Bin
[LightGBM] [Info] Total Bins 2022
[LightGBM] [Info] Number of data points in the train set: 19139, number of used features: 49
[LightGBM] [Info] [binary:BoostFromScore]: pavg=0.400961 -> initscore=-0.401461
[LightGBM] [Info] Start training from score -0.401461
[LightGBM] [Debug] Trained a tree with leaves = 31 and depth = 9
Training until validation scores don't improve for 5 rounds
[LightGBM] [Deb

D:\anaconda3\envs\py3124\Lib\site-packages\lightgbm\basic.py:2521: UserWarning: Overriding the parameters from Reference Dataset.
  _log_warning("Overriding the parameters from Reference Dataset.")


[LightGBM] [Debug] Trained a tree with leaves = 31 and depth = 11
[LightGBM] [Debug] Trained a tree with leaves = 31 and depth = 10
[LightGBM] [Debug] Trained a tree with leaves = 31 and depth = 9
[LightGBM] [Debug] Trained a tree with leaves = 31 and depth = 12
[LightGBM] [Debug] Trained a tree with leaves = 31 and depth = 11
[LightGBM] [Debug] Trained a tree with leaves = 31 and depth = 15
[LightGBM] [Debug] Trained a tree with leaves = 31 and depth = 13
[LightGBM] [Debug] Trained a tree with leaves = 31 and depth = 10
[LightGBM] [Debug] Trained a tree with leaves = 31 and depth = 12
[LightGBM] [Debug] Trained a tree with leaves = 31 and depth = 10
[LightGBM] [Debug] Trained a tree with leaves = 31 and depth = 12
[LightGBM] [Debug] Trained a tree with leaves = 31 and depth = 12
[LightGBM] [Debug] Trained a tree with leaves = 31 and depth = 13
[LightGBM] [Debug] Trained a tree with leaves = 31 and depth = 12
[LightGBM] [Debug] Trained a tree with leaves = 31 and depth = 12
[LightGBM] 

In [27]:
# 常用接口
lgb.best_iteration
lgb.feature_importance()

57

array([ 13,   2,   7,   2,   5,   7,   1,  14,   4,   2,   2,  23,   9,
         1,   4,   2,   2,  13,   3,   4,   3,   1,   1,  19,  44, 129,
        22,  45,  43,  12,  42,  35,  36, 106,  23,  48,  31,  11,  27,
        72,  57, 104,  72,  44, 130, 173,  99, 122,  39])

In [28]:
# 模型保存
lgb.save_model('model.txt')

In [1]:
# 单调约束
my_list= [1, 0, -1] + [0] * 46

params= {'metric': 'auc', 'objective': 'binary'}
params['monotone_constraints']= my_list   # 必须传入所有特征的约束标识
params['monotone_constraints_method']= 'advanced'   # {basic, intermediate, advanced}


lgb = lightgbm.train(params=params, train_set= train1, valid_sets= test1)

print("AUC: ", roc_auc_score(Ytest, lgb.predict(Xtest)))

In [ ]:
# 交互约束
params= {'metric': 'auc', 'objective': 'binary'}
params['interaction_constraints']= [[1, 3, 4], [5, 6]]

lgb = lightgbm.train(params=params, train_set= train1, valid_sets= test1)

print("AUC: ", roc_auc_score(Ytest, lgb.predict(Xtest)))